# S&P 500 — Task-Level Wage Exposure Analysis

Estimates the total wage value of work performed in tasks classified as **E0**, **E1**, and **E2/E3**
and computes the potentially automated wage value under three scenarios (25 %, 35 %, 45 %).

**Pipeline**
1. Load company workforce data (`sp500_company_data.parquet`)
2. Load task-time distribution π (`job_task_time_distribution_30_0.csv`)
3. Load task-level exposure labels (`full_labelset_new.tsv`)
4. Aggregate workforce → occupation-level worker counts and wages
5. Build occupation × task table and attach π values and exposure labels
6. Compute task economic value = workers × wage × π
7. Run diagnostics
8. Aggregate by exposure bucket and apply automation scenarios
9. Save results

In [4]:
# ── Imports ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

print("Libraries loaded ✓")

Libraries loaded ✓


## 1 · Configuration — File Paths

In [5]:
# ── Paths ─────────────────────────────────────────────────────────────────────
BASE = "/Users/nicobagnoli/Documents/PYTHON/SMP500-1"

PARQUET_PATH   = os.path.join(BASE, "data", "sp500_company_data.parquet")
TASK_TIME_PATH = os.path.join(BASE, "data", "job_task_time_distribution_30_0.csv")
EXPOSURE_PATH  = os.path.join(BASE, "Eloundou_New", "full_labelset_new.tsv")
OUTPUT_PATH    = os.path.join(BASE, "Eloundou_New", "task_value_exposure_summary.csv")

for label, path in [("Company data", PARQUET_PATH),
                    ("Task-time dist.", TASK_TIME_PATH),
                    ("Exposure labels", EXPOSURE_PATH)]:
    exists = os.path.exists(path)
    status = "✓" if exists else "✗ MISSING"
    print(f"  {status}  {label:20s}  {path}")
    if not exists:
        raise FileNotFoundError(f"Required file not found: {path}")

  ✓  Company data          /Users/nicobagnoli/Documents/PYTHON/SMP500-1/data/sp500_company_data.parquet
  ✓  Task-time dist.       /Users/nicobagnoli/Documents/PYTHON/SMP500-1/data/job_task_time_distribution_30_0.csv
  ✓  Exposure labels       /Users/nicobagnoli/Documents/PYTHON/SMP500-1/Eloundou_New/full_labelset_new.tsv


## 2 · Load Data

In [6]:
# ── 2.1 Company workforce data ────────────────────────────────────────────────
print("Loading company workforce data …", flush=True)
df_raw = pd.read_parquet(PARQUET_PATH)

print(f"  Rows: {len(df_raw):,}")
print(f"  Columns: {df_raw.columns.tolist()}")

# ── Column auto-detection ─────────────────────────────────────────────────────
REQUIRED = {
    "occ_id"  : ["onet_code", "O*NET-SOC Code", "onet", "soc_code"],
    "wage"    : ["salary", "wage", "annual_wage", "mean_wage"],
    "weight"  : ["weight", "weights", "n_workers", "emp_count"],
}

resolved = {}
for field, candidates in REQUIRED.items():
    found = next((c for c in candidates if c in df_raw.columns), None)
    if found is None:
        raise KeyError(
            f"Cannot find column for '{field}'. "
            f"Expected one of {candidates}. "
            f"Available columns: {df_raw.columns.tolist()}"
        )
    resolved[field] = found
    print(f"  Detected '{field}' → '{found}'")

OCC_COL    = resolved["occ_id"]
WAGE_COL   = resolved["wage"]
WEIGHT_COL = resolved["weight"]

print("\nCompany data loaded ✓")

Loading company workforce data …
  Rows: 16,016,673
  Columns: ['user_id', 'position_id', 'rcid', 'seniority', 'country', 'salary', 'onet_code', 'startdate', 'enddate', 'weight', 'highest_degree', 'sex_predicted', 'ethnicity_predicted', 'ticker', 'naics_code', 'exchange_name', 'company']
  Detected 'occ_id' → 'onet_code'
  Detected 'wage' → 'salary'
  Detected 'weight' → 'weight'

Company data loaded ✓


In [7]:
# ── 2.2 Task-time distribution ────────────────────────────────────────────────
print("Loading task-time distribution …", flush=True)
df_pi = pd.read_csv(TASK_TIME_PATH)

# Auto-detect columns
PI_OCC_CANDIDATES  = ["O*NET-SOC Code", "onet_code", "onet", "soc_code"]
PI_TASK_CANDIDATES = ["Task ID", "task_id", "TaskID"]
PI_PI_CANDIDATES   = ["pi", "Pi", "task_share", "time_share"]

pi_occ_col  = next((c for c in PI_OCC_CANDIDATES  if c in df_pi.columns), None)
pi_task_col = next((c for c in PI_TASK_CANDIDATES if c in df_pi.columns), None)
pi_pi_col   = next((c for c in PI_PI_CANDIDATES   if c in df_pi.columns), None)

for label, val, cands in [
    ("occ",  pi_occ_col,  PI_OCC_CANDIDATES),
    ("task", pi_task_col, PI_TASK_CANDIDATES),
    ("pi",   pi_pi_col,   PI_PI_CANDIDATES),
]:
    if val is None:
        raise KeyError(f"Task-time file: cannot find column for '{label}'. "
                       f"Expected one of {cands}. Got: {df_pi.columns.tolist()}")
    print(f"  Detected '{label}' → '{val}'")

# Rename to standard names for the rest of the notebook
df_pi = df_pi.rename(columns={pi_occ_col: "occ_id", pi_task_col: "task_id", pi_pi_col: "pi"})

print(f"\n  Rows: {len(df_pi):,}")
print(f"  Occupations: {df_pi['occ_id'].nunique():,}")
print(f"  Tasks: {df_pi['task_id'].nunique():,}")
print("\nTask-time distribution loaded ✓")

Loading task-time distribution …
  Detected 'occ' → 'O*NET-SOC Code'
  Detected 'task' → 'Task ID'
  Detected 'pi' → 'pi'

  Rows: 17,890
  Occupations: 894
  Tasks: 17,890

Task-time distribution loaded ✓


In [8]:
# ── 2.3 Task exposure labels ──────────────────────────────────────────────────
print("Loading task exposure labels …", flush=True)
df_exp = pd.read_csv(EXPOSURE_PATH, sep="\t")

# Auto-detect columns
EXP_OCC_CANDIDATES   = ["O*NET-SOC Code", "onet_code", "onet", "soc_code"]
EXP_TASK_CANDIDATES  = ["Task ID", "task_id", "TaskID"]
EXP_LABEL_CANDIDATES = ["new_labels", "exposure_label", "label", "gpt4_exposure", "human_exposure_agg"]

exp_occ_col   = next((c for c in EXP_OCC_CANDIDATES   if c in df_exp.columns), None)
exp_task_col  = next((c for c in EXP_TASK_CANDIDATES  if c in df_exp.columns), None)
exp_label_col = next((c for c in EXP_LABEL_CANDIDATES if c in df_exp.columns), None)

for label, val, cands in [
    ("occ",   exp_occ_col,   EXP_OCC_CANDIDATES),
    ("task",  exp_task_col,  EXP_TASK_CANDIDATES),
    ("label", exp_label_col, EXP_LABEL_CANDIDATES),
]:
    if val is None:
        raise KeyError(f"Exposure file: cannot find column for '{label}'. "
                       f"Expected one of {cands}. Got: {df_exp.columns.tolist()}")
    print(f"  Detected '{label}' → '{val}'")

# Keep only needed columns and rename
df_exp = df_exp[[exp_occ_col, exp_task_col, exp_label_col]].copy()
df_exp.columns = ["occ_id", "task_id", "exposure_label"]

print(f"\n  Rows: {len(df_exp):,}")
print(f"\n  Exposure label distribution:")
print(df_exp["exposure_label"].value_counts().to_string())
print("\nExposure labels loaded ✓")

Loading task exposure labels …
  Detected 'occ' → 'O*NET-SOC Code'
  Detected 'task' → 'Task ID'
  Detected 'label' → 'new_labels'

  Rows: 19,265

  Exposure label distribution:
exposure_label
E0    10206
E1     4677
E2     3327
E3      519

Exposure labels loaded ✓


## 3 · Aggregate Workforce → Occupation-Level Worker Counts and Wages

In [9]:
# ── 3.1  Drop rows missing the occupation identifier ─────────────────────────
before = len(df_raw)
df_work = df_raw.dropna(subset=[OCC_COL]).copy()
after  = len(df_work)
print(f"Dropped {before - after:,} rows with missing occupation code "
      f"({(before - after) / before:.1%} of total)")

# ── 3.2  Impute missing wages from occupation median (weighted) ───────────────
# First compute weighted median salary per onet_code for those that do have one
df_work[WEIGHT_COL] = df_work[WEIGHT_COL].fillna(1.0)     # treat null weight as 1

occ_median_wage = (
    df_work.dropna(subset=[WAGE_COL])
    .groupby(OCC_COL)
    .apply(lambda g: np.average(g[WAGE_COL], weights=g[WEIGHT_COL]))
    .rename("median_occ_wage")
    .reset_index()
)
occ_median_wage.columns = [OCC_COL, "median_occ_wage"]

df_work = df_work.merge(occ_median_wage, on=OCC_COL, how="left")
df_work[WAGE_COL] = df_work[WAGE_COL].fillna(df_work["median_occ_wage"])

n_still_missing = df_work[WAGE_COL].isna().sum()
print(f"After occupation-level imputation: {n_still_missing:,} rows still missing wage "
      f"(these will be excluded from wage mass).")

# ── 3.3  Occupation-level aggregation ────────────────────────────────────────
# n_workers = sum of weights (corrected headcount)
# avg_wage  = weighted mean of salary
def wtd_mean(g):
    mask = g[WAGE_COL].notna()
    if mask.sum() == 0:
        return np.nan
    return np.average(g.loc[mask, WAGE_COL], weights=g.loc[mask, WEIGHT_COL])

occ_agg = (
    df_work.groupby(OCC_COL)
    .apply(lambda g: pd.Series({
        "n_workers": g[WEIGHT_COL].sum(),
        "avg_wage" : wtd_mean(g),
    }))
    .reset_index()
)
occ_agg.columns = ["occ_id", "n_workers", "avg_wage"]

# Drop occupations with no usable wage data
before_occ = len(occ_agg)
occ_agg = occ_agg.dropna(subset=["avg_wage"])
after_occ = len(occ_agg)
print(f"\nOccupations after aggregation : {before_occ:,}  (dropped {before_occ - after_occ:,} with no wage data)")

# ── 3.4  Sanity-check wage mass ───────────────────────────────────────────────
total_wage_mass = (occ_agg["n_workers"] * occ_agg["avg_wage"]).sum()
print(f"\nTotal workforce wage mass  :  ${total_wage_mass:,.0f}")
print(f"Occupations covered        :  {len(occ_agg):,}")
print(f"Weighted workers covered   :  {occ_agg['n_workers'].sum():,.0f}")
print("\nOccupation aggregation ✓")

Dropped 6,712 rows with missing occupation code (0.0% of total)
After occupation-level imputation: 0 rows still missing wage (these will be excluded from wage mass).

Occupations after aggregation : 1,009  (dropped 0 with no wage data)

Total workforce wage mass  :  $1,303,402,913,650
Occupations covered        :  1,009
Weighted workers covered   :  18,649,010

Occupation aggregation ✓


## 4 · Build Occupation × Task Table

In [10]:
# ── 4.1  Join occupation aggregates → task-time table ────────────────────────
# Inner join: only occupations that appear in both the workforce AND the π table
df_occ_task = df_pi.merge(occ_agg, on="occ_id", how="inner")

n_occ_matched   = df_occ_task["occ_id"].nunique()
n_occ_workforce = occ_agg["occ_id"].nunique()
n_occ_pi        = df_pi["occ_id"].nunique()

print(f"Occupations in workforce data  : {n_occ_workforce:,}")
print(f"Occupations in π table         : {n_occ_pi:,}")
print(f"Occupations matched (inner)    : {n_occ_matched:,}")
print(f"Task rows after join            : {len(df_occ_task):,}")

# Wage mass retained after occupation matching
wage_mass_matched = (df_occ_task.groupby("occ_id")
                     .first()[["n_workers", "avg_wage"]]
                     .eval("wage_mass = n_workers * avg_wage")
                     ["wage_mass"].sum())
print(f"\nWage mass in matched occupations  : ${wage_mass_matched:,.0f}")
print(f"Share of total wage mass          : {wage_mass_matched / total_wage_mass:.1%}")

# ── 4.2  Attach exposure labels ───────────────────────────────────────────────
df_occ_task = df_occ_task.merge(df_exp, on=["occ_id", "task_id"], how="left")

n_unlabelled = df_occ_task["exposure_label"].isna().sum()
print(f"\nTask-rows without exposure label  : {n_unlabelled:,} "
      f"({n_unlabelled / len(df_occ_task):.1%})")

# Drop unlabelled tasks (cannot assign exposure bucket)
df_occ_task = df_occ_task.dropna(subset=["exposure_label"])
print(f"Task-rows after dropping unlabelled: {len(df_occ_task):,}")

print("\nOccupation × task table built ✓")

Occupations in workforce data  : 1,009
Occupations in π table         : 894
Occupations matched (inner)    : 887
Task rows after join            : 17,727

Wage mass in matched occupations  : $1,119,205,139,409
Share of total wage mass          : 85.9%

Task-rows without exposure label  : 65 (0.4%)
Task-rows after dropping unlabelled: 17,662

Occupation × task table built ✓


## 5 · Diagnostics

In [11]:
# ── 5.1  π sums per occupation (should all equal 1) ──────────────────────────
pi_sums = df_occ_task.groupby("occ_id")["pi"].sum()

print("=== Task-share (π) sum per occupation ===")
print(f"  Min  : {pi_sums.min():.4f}")
print(f"  Max  : {pi_sums.max():.4f}")
print(f"  Mean : {pi_sums.mean():.4f}")
print(f"  Occupations where |π_sum − 1| > 0.01 : "
      f"{(pi_sums - 1).abs().gt(0.01).sum()}")
print()

# NOTE: Due to unlabelled tasks being dropped, some occupations' π sums may be
#       slightly below 1.  We re-normalise within the labelled task set so that
#       the total task value reconstructs the occupation wage mass.
df_occ_task["pi_norm"] = df_occ_task.groupby("occ_id")["pi"].transform(
    lambda x: x / x.sum()
)
pi_sums_renorm = df_occ_task.groupby("occ_id")["pi_norm"].sum()
print(f"  After re-normalisation — max |π_sum − 1| : "
      f"{(pi_sums_renorm - 1).abs().max():.6f}")

# ── 5.2  Task counts per exposure class ──────────────────────────────────────
print("\n=== Task rows per exposure class ===")
print(df_occ_task["exposure_label"].value_counts().to_string())

print("\nDiagnostics ✓")

=== Task-share (π) sum per occupation ===
  Min  : 0.3246
  Max  : 1.0000
  Mean : 0.9965
  Occupations where |π_sum − 1| > 0.01 : 38

  After re-normalisation — max |π_sum − 1| : 0.000000

=== Task rows per exposure class ===
exposure_label
E0    9706
E1    4324
E2    3133
E3     499

Diagnostics ✓


## 6 · Compute Task Economic Value

In [12]:
# ── Task value = workers × avg_wage × π_norm ─────────────────────────────────
# (pi_norm is the re-normalised share that accounts for tasks with missing labels)
df_occ_task["task_value"] = (
    df_occ_task["n_workers"]
    * df_occ_task["avg_wage"]
    * df_occ_task["pi_norm"]
)

# ── Diagnostic: task values should sum to the occupation wage mass ────────────
reconstructed_wage_mass = df_occ_task["task_value"].sum()

print("=== Wage-mass reconstruction check ===")
print(f"  True occupation wage mass (matched occs)  : ${wage_mass_matched:,.0f}")
print(f"  Sum of task values                         : ${reconstructed_wage_mass:,.0f}")
diff = abs(reconstructed_wage_mass - wage_mass_matched)
print(f"  Absolute difference                        : ${diff:,.0f}  "
      f"({diff / wage_mass_matched:.4%})")
if diff / wage_mass_matched < 1e-6:
    print("  ✓  Task values perfectly reconstruct wage mass")
else:
    print("  ⚠  Small difference expected due to re-normalisation over labelled tasks")

print(f"\nTotal task-value rows  : {len(df_occ_task):,}")
print("\nTask economic values computed ✓")

=== Wage-mass reconstruction check ===
  True occupation wage mass (matched occs)  : $1,119,205,139,409
  Sum of task values                         : $1,119,205,139,409
  Absolute difference                        : $0  (0.0000%)
  ✓  Task values perfectly reconstruct wage mass

Total task-value rows  : 17,662

Task economic values computed ✓


## 7 · Assign Exposure Buckets and Compute Baseline Totals

In [13]:
# ── Map E2 and E3 → E2/E3 bucket ──────────────────────────────────────────────
BUCKET_MAP = {"E0": "E0", "E1": "E1", "E2": "E2/E3", "E3": "E2/E3"}

df_occ_task["bucket"] = df_occ_task["exposure_label"].map(BUCKET_MAP)

unmapped = df_occ_task["bucket"].isna().sum()
if unmapped > 0:
    unknown_labels = df_occ_task.loc[df_occ_task["bucket"].isna(), "exposure_label"].unique()
    print(f"⚠  {unmapped:,} rows have unrecognised exposure labels: {unknown_labels}")
    df_occ_task = df_occ_task.dropna(subset=["bucket"])

# ── Baseline totals per bucket ────────────────────────────────────────────────
baseline = (
    df_occ_task.groupby("bucket")["task_value"]
    .sum()
    .reindex(["E0", "E1", "E2/E3"])
    .fillna(0)
    .rename("baseline_value")
)

total_value = baseline.sum()

print("=== Baseline wage value by exposure class ===")
print(f"{'Class':<10} {'Value ($)':>18} {'Share':>8}")
print("-" * 40)
for bucket, val in baseline.items():
    print(f"{bucket:<10} ${val:>17,.0f} {val / total_value:>7.1%}")
print("-" * 40)
print(f"{'TOTAL':<10} ${total_value:>17,.0f} {'100.0%':>8}")

print("\nExposure bucket totals ✓")

=== Baseline wage value by exposure class ===
Class               Value ($)    Share
----------------------------------------
E0         $  240,810,479,055   21.5%
E1         $  467,033,149,727   41.7%
E2/E3      $  411,361,510,627   36.8%
----------------------------------------
TOTAL      $1,119,205,139,409   100.0%

Exposure bucket totals ✓


## 8 · Automation Scenarios (25 %, 35 %, 45 %)

In [14]:
# ── Automation rates ──────────────────────────────────────────────────────────
SCENARIOS = {"25%": 0.25, "35%": 0.35, "45%": 0.45}

# E0 is never automated
val_E0   = baseline["E0"]
val_E1   = baseline["E1"]
val_E2_3 = baseline["E2/E3"]

rows = []
for label, rate in SCENARIOS.items():
    auto_E1   = val_E1   * rate
    auto_E2_3 = val_E2_3 * rate
    rows.append({
        "Scenario"          : f"Scenario {label}",
        "Rate"              : rate,
        "Auto E1 ($)"       : auto_E1,
        "Auto E2/E3 ($)"    : auto_E2_3,
        "Total Auto ($)"    : auto_E1 + auto_E2_3,
        "Auto Share (%)"    : (auto_E1 + auto_E2_3) / total_value * 100,
    })

df_scenarios = pd.DataFrame(rows)

print("=== Automation scenario results ===\n")
print(df_scenarios.to_string(index=False, float_format="${:,.0f}".format))

print("\nAutomation scenarios ✓")

=== Automation scenario results ===

    Scenario  Rate      Auto E1 ($)   Auto E2/E3 ($)   Total Auto ($)  Auto Share (%)
Scenario 25%    $0 $116,758,287,432 $102,840,377,657 $219,598,665,089             $20
Scenario 35%    $0 $163,461,602,404 $143,976,528,720 $307,438,131,124             $27
Scenario 45%    $0 $210,164,917,377 $185,112,679,782 $395,277,597,159             $35

Automation scenarios ✓


## 9 · Summary Table

In [15]:
# ── Build composite summary table ────────────────────────────────────────────
# Columns: Class | Baseline Value | 25% Automation | 35% Automation | 45% Automation

buckets = ["E0", "E1", "E2/E3"]
summary_rows = []

for b in buckets:
    bval = baseline[b]
    if b == "E0":
        row = {
            "Class"           : b,
            "Baseline Value ($)": bval,
            "25% Automation ($)": 0,
            "35% Automation ($)": 0,
            "45% Automation ($)": 0,
        }
    else:
        col_name = "Auto E1 ($)" if b == "E1" else "Auto E2/E3 ($)"
        sc = df_scenarios.set_index("Scenario")[col_name]
        row = {
            "Class"            : b,
            "Baseline Value ($)": bval,
            "25% Automation ($)": sc["Scenario 25%"],
            "35% Automation ($)": sc["Scenario 35%"],
            "45% Automation ($)": sc["Scenario 45%"],
        }
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)

# ── Print the summary table ───────────────────────────────────────────────────
def fmt_usd(x):
    return f"${x:,.0f}"

print("=" * 90)
print("  TASK-VALUE EXPOSURE SUMMARY  (S&P 500 workforce)")
print("=" * 90)
print(df_summary.to_string(index=False, float_format="${:,.0f}".format))
print("=" * 90)

print(f"\n  Total workforce wage mass represented  : ${total_wage_mass:,.0f}")
print(f"  Wage mass in matched occupations       : ${wage_mass_matched:,.0f}   "
      f"({wage_mass_matched / total_wage_mass:.1%} of total)")
print()

print("  Share of total task value by exposure class:")
for b in buckets:
    print(f"    {b:<6} : {baseline[b] / total_value:>6.1%}")
print()

print("  Share of potentially automated value (E1 + E2/E3):")
for _, srow in df_scenarios.iterrows():
    print(f"    {srow['Scenario']:<16}: {srow['Auto Share (%)']:>5.1f}% of total task value")

print()
print("Summary ✓")

  TASK-VALUE EXPOSURE SUMMARY  (S&P 500 workforce)
Class  Baseline Value ($)  25% Automation ($)  35% Automation ($)  45% Automation ($)
   E0    $240,810,479,055                  $0                  $0                  $0
   E1    $467,033,149,727    $116,758,287,432    $163,461,602,404    $210,164,917,377
E2/E3    $411,361,510,627    $102,840,377,657    $143,976,528,720    $185,112,679,782

  Total workforce wage mass represented  : $1,303,402,913,650
  Wage mass in matched occupations       : $1,119,205,139,409   (85.9% of total)

  Share of total task value by exposure class:
    E0     :  21.5%
    E1     :  41.7%
    E2/E3  :  36.8%

  Share of potentially automated value (E1 + E2/E3):
    Scenario 25%    :  19.6% of total task value
    Scenario 35%    :  27.5% of total task value
    Scenario 45%    :  35.3% of total task value

Summary ✓


## 10 · Top Occupations Contributing to E1 and E2/E3 Exposure

In [16]:
# Build occupation-level task-value aggregates with titles where available
occ_bucket_value = (
    df_occ_task.groupby(["occ_id", "bucket"])["task_value"]
    .sum()
    .reset_index()
)

# Attach occupation title from the π table (if it has a Title column)
title_map = None
if "Title" in df_pi.columns or "title" in df_pi.columns:
    title_col = "Title" if "Title" in df_pi.columns else "title"
    title_map = df_pi[["occ_id", title_col]].drop_duplicates().rename(columns={title_col: "occ_title"})
    occ_bucket_value = occ_bucket_value.merge(title_map, on="occ_id", how="left")
else:
    occ_bucket_value["occ_title"] = occ_bucket_value["occ_id"]

TOP_N = 15

for bucket in ["E1", "E2/E3"]:
    sub = (
        occ_bucket_value[occ_bucket_value["bucket"] == bucket]
        .sort_values("task_value", ascending=False)
        .head(TOP_N)
        .reset_index(drop=True)
    )
    sub.index += 1
    sub["Share of bucket (%)"] = sub["task_value"] / baseline[bucket] * 100
    print(f"\n{'='*65}")
    print(f"  Top {TOP_N} occupations by task value in  {bucket}")
    print(f"{'='*65}")
    cols = ["occ_id", "occ_title", "task_value", "Share of bucket (%)"]
    print(sub[cols].to_string(float_format="${:,.0f}".format))


  Top 15 occupations by task value in  E1
        occ_id                                        occ_title      task_value  Share of bucket (%)
1   15-1252.00                              Software Developers $89,635,222,291                  $19
2   11-2021.00                               Marketing Managers $34,249,275,530                   $7
3   15-1299.08            Computer Systems Engineers/Architects $31,037,207,834                   $7
4   15-1299.09          Information Technology Project Managers $20,277,318,146                   $4
5   15-2051.01                   Business Intelligence Analysts $15,306,839,363                   $3
6   11-2022.00                                   Sales Managers $13,067,796,120                   $3
7   15-1243.00                              Database Architects $11,733,003,864                   $3
8   11-1011.00                                 Chief Executives $10,557,724,051                   $2
9   11-3031.03                         Investmen